# Encoder Ablation Study

Tests whether the downstream RAGNN benefits from the encoder's latent graph by comparing three conditions against the trained baseline:

| Condition | GNN input |
|-----------|----------|
| `baseline` | Real encoder posterior (trained model) |
| `full` | Uniform weight across all non-null edge types (all edges, equal weight) |
| `empty` | All weight on null type → self-loops only (MLP-like) |
| `random` | Fresh random soft posterior at every timestep |

**Workflow:**
1. Discover all trial directories; auto-select the best checkpoint per seed.
2. For each seed: run all conditions on the same set of chronics and print a survival table.
3. Compute the Spearman rank-order correlation of condition rankings across seeds (measures whether seeds agree on which conditions are better).
4. Plot boxplots of survived-% per condition per seed.

In [1]:
%matplotlib inline
import logging
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr
from tabulate import tabulate

logging.basicConfig(level=logging.WARNING)
plt.style.use("default")

REPO_ROOT = Path("/home/adrian/Dev/NRI-for-explainable-RL-in-Power-Grids").resolve()
os.chdir(REPO_ROOT)
for p in [str(REPO_ROOT / "src"), str(REPO_ROOT / "experiments"), str(REPO_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from analysis.cross_seed_analysis import find_trial_dirs, get_seed, best_checkpoint, load_agent
from analysis.ablation_encoder import ablated_encoder, run_episodes, summarise
plt.style.use("default")

In [2]:
# ── experiment ─────────────────────────────────────────────────────────────────
EXPERIMENT_DIR = REPO_ROOT / "results/2026_07_20_IEEE14/rappo_multiseed_gcnconv"

# ── environment ────────────────────────────────────────────────────────────────
ENV_NAME   = "l2rpn_case14_sandbox_test"
N_CHRONICS = 10      # number of test chronics to run per condition per seed
MAX_ITER   = None    # max steps per episode (None = full episode)

# ── ablation ───────────────────────────────────────────────────────────────────
CONDITIONS = ["baseline", "full", "empty", "random"]

# ── checkpoint ─────────────────────────────────────────────────────────────────
CONV_TYPE = None     # None → from checkpoint config; "gin" for pre-revert ckpts

In [3]:
import ray

ray.init(
    ignore_reinit_error=True,
    logging_level=logging.ERROR,
    log_to_driver=False,
    num_cpus=1,
    object_store_memory=512 * 1024 * 1024,
)

trial_dirs = find_trial_dirs(EXPERIMENT_DIR)
print(f"Found {len(trial_dirs)} trial dir(s) in {EXPERIMENT_DIR}")

seed_to_info: dict[int, tuple[Path, str]] = {}
for td in trial_dirs:
    try:
        seed = get_seed(td)
    except KeyError as e:
        print(f"  WARNING: skipping {td.name} — {e}")
        continue
    try:
        ckpt = best_checkpoint(td)
    except FileNotFoundError as e:
        print(f"  WARNING: skipping {td.name} — {e}")
        continue
    seed_to_info[seed] = (td, ckpt)
    print(f"  seed={seed}  checkpoint={ckpt}  trial={td.name}")

seeds = sorted(seed_to_info.keys())
print(f"\nSeeds: {seeds}")

/home/adrian/.conda/envs/L2RPN/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-10 17:20:35,696	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Found 5 trial dir(s) in /home/adrian/Dev/NRI-for-explainable-RL-in-Power-Grids/results/2026_07_20_IEEE14/rappo_multiseed_gcnconv
  seed=0  checkpoint=checkpoint_000008  trial=CustomPPO_RARL_5887369_1b111_2026-07-18_21-30-16
  seed=1  checkpoint=checkpoint_000007  trial=CustomPPO_RARL_5887370_1ae2e_2026-07-18_21-30-16
  seed=2  checkpoint=checkpoint_000007  trial=CustomPPO_RARL_5887371_15dd9_2026-07-18_21-30-08
  seed=3  checkpoint=checkpoint_000007  trial=CustomPPO_RARL_5887372_1a964_2026-07-18_21-30-16
  seed=4  checkpoint=checkpoint_000006  trial=CustomPPO_RARL_5887373_1c693_2026-07-18_21-30-19

Seeds: [0, 1, 2, 3, 4]


## Per-Seed Ablation

For each seed: load the agent, run every condition on the same chronic set, print the results table, and record the per-condition survival scores for later cross-seed Spearman analysis.

In [4]:
# seed → {condition → summary dict}
all_results: dict[int, dict[str, dict]] = {}
# seed → {condition → list of per-episode dicts}
all_episodes: dict[int, dict[str, list]] = {}
gym_wrappers = []  # keep alive

for seed in seeds:
    trial_dir, ckpt_name = seed_to_info[seed]
    print(f"\n{'═'*60}")
    print(f"Seed {seed}  |  {ckpt_name}  |  {trial_dir.name}")
    print(f"{'═'*60}")

    agent, g2op_env, gym_wrapper = load_agent(
        trial_dir, ckpt_name, ENV_NAME, conv_type=CONV_TYPE
    )
    gym_wrappers.append(gym_wrapper)

    feature_extractor = agent._rllib_agent.model.ragnn
    chronic_ids = list(range(min(N_CHRONICS, len(g2op_env.chronics_handler.subpaths))))

    seed_results: dict[str, dict] = {}
    seed_episodes: dict[str, list] = {}
    for condition in CONDITIONS:
        with ablated_encoder(feature_extractor, condition):
            episodes = run_episodes(agent, g2op_env, chronic_ids, MAX_ITER)
        seed_results[condition] = summarise(episodes)
        seed_episodes[condition] = episodes

    all_results[seed] = seed_results
    all_episodes[seed] = seed_episodes

    # ── per-seed table ────────────────────────────────────────────────────────
    baseline_mean = seed_results["baseline"]["mean_steps"]
    rows = []
    for cond in CONDITIONS:
        s = seed_results[cond]
        delta = s["mean_steps"] - baseline_mean
        rows.append([
            cond,
            f"{s['mean_steps']:.0f}",
            f"{s['median_steps']:.0f}",
            f"{s['survived_pct']:.1f}",
            f"{s['completed_pct']:.1f}",
            f"{delta:+.0f}" if cond != "baseline" else "—",
        ])

    headers = ["Condition", "Mean steps", "Median steps", "Survived %", "Completed %", "Δ mean steps"]
    print(tabulate(rows, headers=headers, tablefmt="github"))


════════════════════════════════════════════════════════════
Seed 0  |  checkpoint_000008  |  CustomPPO_RARL_5887369_1b111_2026-07-18_21-30-16
════════════════════════════════════════════════════════════


/home/adrian/.conda/envs/L2RPN/lib/python3.10/site-packages/grid2op/MakeEnv/Make.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


KeyboardInterrupt: 

## Cross-Seed Spearman Rank Correlation

For each seed we rank the ablation conditions by `survived_pct` (higher = better rank).
We then compute the pairwise Spearman rank-order correlation between seeds.

- **ρ → 1**: seeds agree that the same conditions are better/worse — consistent signal.
- **ρ → 0**: seeds disagree on the relative ordering of conditions.

In [ ]:
# ── build per-seed score vectors (survived_pct, ordered by CONDITIONS) ─────────
score_matrix = np.array(
    [[all_results[s][c]["survived_pct"] for c in CONDITIONS] for s in seeds]
)  # [S, C]

# ── pairwise Spearman r ────────────────────────────────────────────────────────
S = len(seeds)
rho_matrix = np.eye(S)
pval_matrix = np.zeros((S, S))

for i in range(S):
    for j in range(i + 1, S):
        r, p = spearmanr(score_matrix[i], score_matrix[j])
        rho_matrix[i, j] = rho_matrix[j, i] = float(r)
        pval_matrix[i, j] = pval_matrix[j, i] = float(p)

# ── print as table ─────────────────────────────────────────────────────────────
print("\nSpearman ρ of condition ranking (survived_pct) across seeds:\n")
seed_labels = [f"seed {s}" for s in seeds]
rho_rows = [[seed_labels[i]] + [f"{rho_matrix[i, j]:.3f}" for j in range(S)] for i in range(S)]
print(tabulate(rho_rows, headers=[""] + seed_labels, tablefmt="github"))

print("\np-values:\n")
pval_rows = [[seed_labels[i]] + [f"{pval_matrix[i, j]:.3f}" for j in range(S)] for i in range(S)]
print(tabulate(pval_rows, headers=[""] + seed_labels, tablefmt="github"))

# ── summary stats ──────────────────────────────────────────────────────────────
off_diag = rho_matrix[np.triu_indices(S, k=1)]
print(f"\nOff-diagonal Spearman ρ — mean={off_diag.mean():.3f}  "
      f"min={off_diag.min():.3f}  max={off_diag.max():.3f}")

print("\nPer-seed condition ranking (by survived_pct, best→worst):")
rank_rows = []
for i, s in enumerate(seeds):
    order = np.argsort(score_matrix[i])[::-1]
    rank_rows.append([f"seed {s}"] + [CONDITIONS[k] for k in order])
print(tabulate(rank_rows, headers=["Seed"] + [f"#{r+1}" for r in range(len(CONDITIONS))], tablefmt="github"))

In [ ]:
# ── Spearman ρ heatmap ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(max(4, S * 1.1), max(3.5, S * 1.0)))
im = ax.imshow(rho_matrix, vmin=-1, vmax=1, cmap="RdYlGn")
plt.colorbar(im, ax=ax, label="Spearman ρ")
ax.set_xticks(range(S))
ax.set_yticks(range(S))
ax.set_xticklabels(seed_labels, rotation=30, ha="right")
ax.set_yticklabels(seed_labels)
for i in range(S):
    for j in range(S):
        ax.text(j, i, f"{rho_matrix[i, j]:.2f}", ha="center", va="center", fontsize=9)
ax.set_title("Cross-seed Spearman ρ of ablation condition ranking\n(survived_pct)")
plt.tight_layout()
plt.show()

## Survival Distribution per Condition per Seed

In [ ]:
COLORS = {"baseline": "#2ca02c", "full": "#1f77b4", "empty": "#d62728", "random": "#ff7f0e"}
LABELS = {
    "baseline": "Baseline\n(encoder)",
    "full":     "Full\n(all edges)",
    "empty":    "Empty\n(self-loops)",
    "random":   "Random\n(noise)",
}

fig, axes = plt.subplots(1, S, figsize=(4 * S, 5), sharey=True)
if S == 1:
    axes = [axes]

for ax, seed in zip(axes, seeds):
    data = [
        [ep["survived_pct"] for ep in all_episodes[seed][c]]
        for c in CONDITIONS
    ]
    bp = ax.boxplot(
        data,
        patch_artist=True,
        widths=0.55,
        medianprops=dict(color="black", linewidth=2),
        flierprops=dict(marker=".", markersize=3, alpha=0.4),
    )
    for patch, cond in zip(bp["boxes"], CONDITIONS):
        patch.set_facecolor(COLORS[cond])
        patch.set_alpha(0.75)

    ax.set_xticks(range(1, len(CONDITIONS) + 1))
    ax.set_xticklabels([LABELS[c] for c in CONDITIONS], fontsize=9)
    ax.set_title(f"Seed {seed}")
    ax.set_ylim(-2, 105)
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("Survived % of episode")
fig.suptitle(
    "Encoder ablation — survival distribution per condition per seed\n"
    f"({N_CHRONICS} chronics, {ENV_NAME})",
    fontsize=12,
)
plt.tight_layout()

out_path = EXPERIMENT_DIR / "ablation_encoder_multiseed.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()

In [ ]:
ray.shutdown()